# Zgodovinske značilke

Nalozimo osnovne znacilke

In [2]:
import pandas as pd
import numpy as np

from collections import defaultdict, deque
from pathlib import Path

features = pd.read_csv("./data/processed/osnovne_znacilke.csv")

features["tourney_date"] = pd.to_datetime(features["tourney_date"])
features["year"] = features["tourney_date"].dt.year

features = features.sort_values(
    ["tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

features.head()

,tourney_id,tourney_name,tourney_date,year,match_num,p1_id,p1_name,p2_id,p2_name,target,...,tourney_level,best_of,round,hand_matchup,rank_diff,rank_points_diff,age_diff,height_diff,round_num,draw_size
0,2000-339,Adelaide,2000-01-03,2000,1,103096,Arnaud Clement,102358,Thomas Enqvist,0,...,A,3,R32,R_vs_R,52.0,-1801.0,-3.7,-17.0,3,32
1,2000-339,Adelaide,2000-01-03,2000,2,103819,Roger Federer,102533,Jens Knippschild,1,...,A,3,R32,R_vs_R,-27.0,224.0,-6.5,-5.0,3,32
2,2000-339,Adelaide,2000-01-03,2000,3,101885,Wayne Arthurs,102998,Jan Michael Gambill,0,...,A,3,R32,L_vs_R,47.0,-354.0,6.2,0.0,3,32
3,2000-339,Adelaide,2000-01-03,2000,4,102776,Andrew Ilie,103206,Sebastien Grosjean,0,...,A,3,R32,R_vs_R,27.0,-453.0,2.1,5.0,3,32
4,2000-339,Adelaide,2000-01-03,2000,5,102796,Magnus Norman,102401,Scott Draper,1,...,A,3,R32,R_vs_L,-139.0,1451.0,-2.0,10.0,3,32


Dodamo osnovne funkcije in konstante

In [4]:
def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def update_elo(rating_a, rating_b, score_a, k=32):
    """
    score_a = 1, če igralec A zmaga
    score_a = 0, če igralec A izgubi
    """
    exp_a = expected_score(rating_a, rating_b)
    exp_b = 1 - exp_a

    score_b = 1 - score_a

    new_a = rating_a + k * (score_a - exp_a)
    new_b = rating_b + k * (score_b - exp_b)

    return new_a, new_b


def winrate_from_history(history, default=0.5):
    if len(history) == 0:
        return default
    return np.mean(history)


def safe_rate(wins, matches, default=0.5):
    if matches == 0:
        return default
    return wins / matches

In [5]:
INITIAL_ELO = 1500.0
K_FACTOR = 32

In [6]:
# Elo rating igralca
elo = defaultdict(lambda: INITIAL_ELO)

# Elo rating igralca na posamezni površini
surface_elo = defaultdict(lambda: INITIAL_ELO)

# Zadnjih 10 rezultatov igralca
last10_results = defaultdict(lambda: deque(maxlen=10))

# Zadnjih 10 rezultatov igralca na posamezni površini
surface_last10_results = defaultdict(lambda: deque(maxlen=10))

# Skupni rezultati
total_matches = defaultdict(int)
total_wins = defaultdict(int)

# Rezultati po površinah
surface_matches = defaultdict(int)
surface_wins = defaultdict(int)

historical_rows = []

# Pomembno:
# Ker tourney_date pri Sackmannu pogosto pomeni začetek tedna turnirja,
# vse tekme z istim datumom najprej dobijo značilke,
# šele potem posodobimo ratinge.
for date, group in features.groupby("tourney_date", sort=True):

    pending_updates = []

    for idx, row in group.iterrows():
        p1 = row["p1_id"]
        p2 = row["p2_id"]
        surface = row["surface"]
        target = int(row["target"])

        p1_surface_key = (p1, surface)
        p2_surface_key = (p2, surface)

        # -------------------------
        # Vrednosti PRED tekmo
        # -------------------------

        p1_elo = elo[p1]
        p2_elo = elo[p2]

        p1_surface_elo = surface_elo[p1_surface_key]
        p2_surface_elo = surface_elo[p2_surface_key]

        p1_last10 = winrate_from_history(last10_results[p1])
        p2_last10 = winrate_from_history(last10_results[p2])

        p1_surface_last10 = winrate_from_history(surface_last10_results[p1_surface_key])
        p2_surface_last10 = winrate_from_history(surface_last10_results[p2_surface_key])

        p1_total_winrate = safe_rate(total_wins[p1], total_matches[p1])
        p2_total_winrate = safe_rate(total_wins[p2], total_matches[p2])

        p1_surface_winrate = safe_rate(
            surface_wins[p1_surface_key],
            surface_matches[p1_surface_key]
        )
        p2_surface_winrate = safe_rate(
            surface_wins[p2_surface_key],
            surface_matches[p2_surface_key]
        )

        p1_matches_played = total_matches[p1]
        p2_matches_played = total_matches[p2]

        p1_surface_matches_played = surface_matches[p1_surface_key]
        p2_surface_matches_played = surface_matches[p2_surface_key]

        historical_rows.append({
            "row_id": idx,

            "p1_elo": p1_elo,
            "p2_elo": p2_elo,
            "elo_diff": p1_elo - p2_elo,

            "p1_surface_elo": p1_surface_elo,
            "p2_surface_elo": p2_surface_elo,
            "surface_elo_diff": p1_surface_elo - p2_surface_elo,

            "p1_last10_winrate": p1_last10,
            "p2_last10_winrate": p2_last10,
            "last10_winrate_diff": p1_last10 - p2_last10,

            "p1_surface_last10_winrate": p1_surface_last10,
            "p2_surface_last10_winrate": p2_surface_last10,
            "surface_last10_winrate_diff": p1_surface_last10 - p2_surface_last10,

            "p1_total_winrate": p1_total_winrate,
            "p2_total_winrate": p2_total_winrate,
            "total_winrate_diff": p1_total_winrate - p2_total_winrate,

            "p1_surface_winrate": p1_surface_winrate,
            "p2_surface_winrate": p2_surface_winrate,
            "surface_winrate_diff": p1_surface_winrate - p2_surface_winrate,

            "p1_matches_played": p1_matches_played,
            "p2_matches_played": p2_matches_played,
            "matches_played_diff": p1_matches_played - p2_matches_played,

            "p1_surface_matches_played": p1_surface_matches_played,
            "p2_surface_matches_played": p2_surface_matches_played,
            "surface_matches_played_diff": (
                p1_surface_matches_played - p2_surface_matches_played
            )
        })

        pending_updates.append((p1, p2, surface, target))

    # -------------------------
    # Posodobitve PO tekmah tega datuma
    # -------------------------

    for p1, p2, surface, target in pending_updates:
        p1_score = target
        p2_score = 1 - target

        p1_surface_key = (p1, surface)
        p2_surface_key = (p2, surface)

        # update splošnega Elo
        new_p1_elo, new_p2_elo = update_elo(
            elo[p1],
            elo[p2],
            p1_score,
            k=K_FACTOR
        )

        elo[p1] = new_p1_elo
        elo[p2] = new_p2_elo

        # update surface Elo
        new_p1_surface_elo, new_p2_surface_elo = update_elo(
            surface_elo[p1_surface_key],
            surface_elo[p2_surface_key],
            p1_score,
            k=K_FACTOR
        )

        surface_elo[p1_surface_key] = new_p1_surface_elo
        surface_elo[p2_surface_key] = new_p2_surface_elo

        # update forme
        last10_results[p1].append(p1_score)
        last10_results[p2].append(p2_score)

        surface_last10_results[p1_surface_key].append(p1_score)
        surface_last10_results[p2_surface_key].append(p2_score)

        # update skupnih statistik
        total_matches[p1] += 1
        total_matches[p2] += 1

        total_wins[p1] += p1_score
        total_wins[p2] += p2_score

        # update statistik po površini
        surface_matches[p1_surface_key] += 1
        surface_matches[p2_surface_key] += 1

        surface_wins[p1_surface_key] += p1_score
        surface_wins[p2_surface_key] += p2_score

In [13]:
historical_df = pd.DataFrame(historical_rows).set_index("row_id")

historical_df.tail()

,p1_elo,p2_elo,elo_diff,p1_surface_elo,p2_surface_elo,surface_elo_diff,p1_last10_winrate,p2_last10_winrate,last10_winrate_diff,p1_surface_last10_winrate,...,total_winrate_diff,p1_surface_winrate,p2_surface_winrate,surface_winrate_diff,p1_matches_played,p2_matches_played,matches_played_diff,p1_surface_matches_played,p2_surface_matches_played,surface_matches_played_diff
row_id,,,,,,,,,,,,,,,,,,,,,
65209,1782.544751,1531.660967,250.883784,1714.843438,1524.077189,190.766248,0.700000,0.428571,0.271429,0.700000,...,0.145897,0.600000,0.428571,0.171429,94,7,87,55,7,48
65210,1782.544751,1740.520200,42.024551,1714.843438,1718.661352,-3.817915,0.700000,0.600000,0.100000,0.700000,...,-0.057111,0.600000,0.680000,-0.080000,94,38,56,55,25,30
65211,1678.961646,1531.660967,147.300679,1647.753878,1524.077189,123.676688,0.500000,0.428571,0.071429,0.500000,...,0.071429,0.510204,0.428571,0.081633,74,7,67,49,7,42
65212,1510.412656,1489.067557,21.345098,1500.000000,1486.346728,13.653272,0.400000,0.200000,0.200000,0.500000,...,0.083333,0.500000,0.300000,0.200000,12,54,-42,0,30,-30
65213,1531.660967,1510.412656,21.248311,1524.077189,1500.000000,24.077189,0.428571,0.400000,0.028571,0.428571,...,0.011905,0.428571,0.500000,-0.071429,7,12,-5,7,0,7


In [14]:
features_final = features.join(historical_df)

features_final.tail()

,tourney_id,tourney_name,tourney_date,year,match_num,p1_id,p1_name,p2_id,p2_name,target,...,total_winrate_diff,p1_surface_winrate,p2_surface_winrate,surface_winrate_diff,p1_matches_played,p2_matches_played,matches_played_diff,p1_surface_matches_played,p2_surface_matches_played,surface_matches_played_diff
65209,2024-7696,Next Gen Finals,2024-12-18,2024,396,209950,Arthur Fils,210530,Learner Tien,0,...,0.145897,0.600000,0.428571,0.171429,94,7,87,55,7,48
65210,2024-7696,Next Gen Finals,2024-12-18,2024,397,209950,Arthur Fils,210150,Jakub Mensik,1,...,-0.057111,0.600000,0.680000,-0.080000,94,38,56,55,25,30
65211,2024-7696,Next Gen Finals,2024-12-18,2024,398,210506,Alex Michelsen,210530,Learner Tien,0,...,0.071429,0.510204,0.428571,0.081633,74,7,67,49,7,42
65212,2024-7696,Next Gen Finals,2024-12-18,2024,399,211663,Joao Fonseca,209414,Luca Van Assche,1,...,0.083333,0.500000,0.300000,0.200000,12,54,-42,0,30,-30
65213,2024-7696,Next Gen Finals,2024-12-18,2024,400,210530,Learner Tien,211663,Joao Fonseca,0,...,0.011905,0.428571,0.500000,-0.071429,7,12,-5,7,0,7


In [15]:
print("features_basic:", features.shape)
print("historical features:", historical_df.shape)
print("features_final:", features_final.shape)

features_basic: (65214, 21)
historical features: (65214, 24)
features_final: (65214, 45)


In [16]:
history_cols = historical_df.columns.tolist()

features_final[history_cols].isna().sum().sort_values(ascending=False).head(20)

p1_elo                         0
p2_elo                         0
p2_surface_matches_played      0
p1_surface_matches_played      0
matches_played_diff            0
p2_matches_played              0
p1_matches_played              0
surface_winrate_diff           0
p2_surface_winrate             0
p1_surface_winrate             0
total_winrate_diff             0
p2_total_winrate               0
p1_total_winrate               0
surface_last10_winrate_diff    0
p2_surface_last10_winrate      0
p1_surface_last10_winrate      0
last10_winrate_diff            0
p2_last10_winrate              0
p1_last10_winrate              0
surface_elo_diff               0
dtype: int64

In [17]:
features_final[[
    "elo_diff",
    "surface_elo_diff",
    "last10_winrate_diff",
    "surface_winrate_diff",
    "target"
]].corr()["target"].sort_values(ascending=False)

target                  1.000000
elo_diff                0.391643
surface_elo_diff        0.376966
surface_winrate_diff    0.307660
last10_winrate_diff     0.302677
Name: target, dtype: float64

In [18]:
features_final["p1_higher_elo"] = (
    features_final["p1_elo"] > features_final["p2_elo"]
).astype(int)

features_final.groupby("p1_higher_elo")["target"].mean()

p1_higher_elo
0    0.336262
1    0.658606
Name: target, dtype: float64

In [21]:
clay_examples = features_final[
    features_final["surface"] == "Clay"
].copy()

clay_examples[[
    "tourney_date",
    "p1_name",
    "p2_name",
    "surface",
    "p1_surface_elo",
    "p2_surface_elo",
    "surface_elo_diff",
    "target"
]].sort_values("surface_elo_diff", ascending=False).head(10)

,tourney_date,p1_name,p2_name,surface,p1_surface_elo,p2_surface_elo,surface_elo_diff,target
38690,2014-02-17,Rafael Nadal,Joao Sousa,Clay,2309.083295,1458.708219,850.375076,1
39456,2014-05-26,Rafael Nadal,Dusan Lajovic,Clay,2245.707609,1422.874647,822.832962,1
34391,2012-05-27,Rafael Nadal,Denis Istomin,Clay,2267.112557,1445.311845,821.800712,1
41229,2015-02-23,Rafael Nadal,Facundo Arguello,Clay,2240.823565,1427.873397,812.950168,1
46682,2017-04-24,Rafael Nadal,Rogerio Dutra Silva,Clay,2148.278579,1342.927381,805.351199,1
31628,2011-05-08,Rafael Nadal,Paolo Lorenzi,Clay,2258.397371,1454.123887,804.273484,1
36909,2013-05-27,Rafael Nadal,Martin Klizan,Clay,2290.185400,1503.546441,786.638959,1
26570,2009-05-25,Rafael Nadal,Teymuraz Gabashvili,Clay,2239.347357,1453.867228,785.480129,1
31472,2011-04-18,Rafael Nadal,Ivan Dodig,Clay,2270.859326,1491.606364,779.252962,1
41175,2015-02-16,Rafael Nadal,Pablo Carreno Busta,Clay,2267.803615,1494.084761,773.718854,1


In [ ]:
output_path = Path("./data/processed/koncne_znacilke.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

features_final.to_csv(output_path, index=False)

print(features_final.shape)

Saved to data/processed/koncne_znacilke.csv
(65214, 46)
